# 📊 Evaluation Lab — OA · mIoU · per-class F1 · confusion matrices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/%URL%)
[![GitHub](https://img.shields.io/badge/Repo-AnyDisasterMapping-181717?logo=github)](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping)
![Auto](https://img.shields.io/badge/runs-automatically-3fb950)
![Task](https://img.shields.io/badge/task-seg%20%2B%20cd-evaluation-4c8cbf) ![Mirror](https://img.shields.io/badge/matches-test.py%20aggregation-3fb950)
Drop any prediction + ground-truth pair (PNG/TIF/NPY) and get the framework's test.py-grade metrics instantly: overall accuracy, mIoU, per-class IoU / precision / recall / F1 (Dice), support, and the full **confusion matrix** — for both `seg` (multi-class) and `cd` (binary change) conventions. No synthetic pair? A built-in error-model generator produces a controllable-quality pair so the metric suite is always demonstrable.


In [ ]:
#@title ⚙️ SETUP — auto-installs, idempotent {display-mode:"form"}
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("prettytable", "pillow", "numpy", "matplotlib")
import os, numpy as np
NOTEBOOK = "evaluation_metrics_colab.ipynb"
STATE = {}
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
print("✅ setup complete")

In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 📤 UPLOAD — prediction + ground truth (or synthetic pair with error model) {display-mode:"form"}
TASK = "seg"  #@param ["seg", "cd"]
N_CLASSES = 5  #@param {type:"integer"}
PIXEL_ERROR_RATE = 0.12  #@param {type:"slider", min:0.0, max:0.5, step:0.01}
SEED = 42  #@param {type:"integer"}

import io, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def read_mask():
    print("📂 file picker — choose a mask (png / tif / npy); Cancel for synthetic")
    try:
        from google.colab import files as _f
        up = _f.upload()
        for nm, data in up.items():
            if nm.endswith(".npy"):
                return np.load(io.BytesIO(data)), nm
            im = Image.open(io.BytesIO(data)).convert("L")
            return np.asarray(im), nm
    except Exception:
        pass
    return None, None

gt, gt_name = read_mask()
pred, pred_name = read_mask()

if gt is None or pred is None:
    rng = np.random.default_rng(SEED)
    size = 256
    yy, xx = np.mgrid[0:size, 0:size]
    gt = np.zeros((size, size), np.uint8)
    gt[yy < 86] = 1
    gt[(yy >= 86) & (xx > 128)] = 2
    gt[(yy >= 86) & (xx <= 128) & (xx > 64)] = 3
    gt[(yy >= 86) & (xx <= 64)] = 4 if TASK == "seg" else 1
    if TASK == "cd":
        gt = (gt > 0).astype(np.uint8)          # binary change mask
    noise = rng.random(gt.shape) < PIXEL_ERROR_RATE
    wrong = rng.integers(0, (2 if TASK == "cd" else N_CLASSES), size=gt.shape)
    pred = np.where(noise, wrong, gt).astype(np.uint8)
    gt_name, pred_name = "synthetic_gt.png", f"synthetic_pred@{PIXEL_ERROR_RATE:.0%}_error.png"
    print(f"🛟 synthetic pair generated — task={TASK}, error rate={PIXEL_ERROR_RATE:.0%}")
else:
    print("✅ uploaded:", pred_name, "vs", gt_name)

if gt.shape != pred.shape:
    from PIL import Image as _Img
    print(f"⚠️ shape mismatch {pred.shape} vs {gt.shape} — resizing prediction (nearest)")
    pred = np.asarray(_Img.fromarray(pred).resize((gt.shape[1], gt.shape[0]), _Img.NEAREST))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), constrained_layout=True)
axes[0].imshow(gt, cmap="tab10", interpolation="nearest"); axes[0].set_title(f"ground truth — {gt_name}"); axes[0].axis("off")
axes[1].imshow(pred, cmap="tab10", interpolation="nearest"); axes[1].set_title(f"prediction — {pred_name}"); axes[1].axis("off")
plt.show()


In [ ]:
#@title ▶️ RUN — full metric suite: OA · mIoU · per-class P/R/F1 · Dice · confusion {display-mode:"form"}
import json, numpy as np
import matplotlib.pyplot as plt
from prettytable import PrettyTable

K = int(max(gt.max(), pred.max())) + 1 if TASK == "seg" else 2
cm = np.zeros((K, K), np.int64)
for g, p in zip(gt.ravel(), pred.ravel()):
    cm[g, p] += 1

OA = np.trace(cm) / cm.sum()
inter = np.diag(cm)
union = cm.sum(0) + cm.sum(1) - inter
iou = np.divide(inter, union, out=np.zeros_like(inter, dtype=float), where=union > 0)
prec = np.divide(cm.sum(1), cm.sum(0), out=np.zeros(K), where=cm.sum(0) > 0)
rec = inter / np.maximum(cm.sum(1), 1)
f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
dice = f1
mIoU = iou[1:].mean() if TASK == "cd" else iou.mean()

names = (["no-change", "change"] if TASK == "cd" else [f"class {i}" for i in range(K)])
t = PrettyTable()
t.field_names = ["class", "IoU", "precision", "recall", "F1/Dice", "support"]
t.align = "l"; t.float_format = ".3"
for i in range(K):
    t.add_row([names[i], iou[i], prec[i], rec[i], f1[i], int(cm[i].sum())])
print(t.get_string())
print(f"\n🏅 OVERALL  OA = {OA:.4f} · mIoU = {mIoU:.4f}"
      + (f" · change-F1 = {f1[1]:.4f}" if TASK == "cd" else ""))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(K), names, rotation=45, fontsize=8)
axes[0].set_yticks(range(K), names, fontsize=8)
for i in range(K):
    for j in range(K):
        if cm[i, j]:
            axes[0].text(j, i, cm[i, j], ha="center", va="center",
                         color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("ground truth")
axes[0].set_title("confusion matrix")
plt.colorbar(im, ax=axes[0], fraction=0.046)

xpos = np.arange(K)
axes[1].bar(xpos - 0.2, iou, 0.4, label="IoU")
axes[1].bar(xpos + 0.2, f1, 0.4, label="F1/Dice")
axes[1].set_xticks(xpos, names, fontsize=8)
axes[1].set_ylim(0, 1); axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
axes[1].set_title("per-class IoU vs F1")
for xi, v in zip(xpos, iou):
    axes[1].text(xi - 0.2, v + 0.02, f"{v:.2f}", ha="center", fontsize=7)

fig.suptitle(f"evaluation — {TASK.upper()} · OA {OA:.3f} · mIoU {mIoU:.3f}")
plt.show()

METRICS = {"task": TASK, "OA": float(OA), "mIoU": float(mIoU),
           "per_class": {names[i]: {"iou": float(iou[i]), "precision": float(prec[i]),
                                    "recall": float(rec[i]), "f1": float(f1[i]),
                                    "support": int(cm[i].sum())} for i in range(K)},
           "confusion_matrix": cm.tolist()}
json.dump(METRICS, open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2)
STATE.update(task=TASK, OA=round(float(OA), 4), mIoU=round(float(mIoU), 4))
print("💾 metrics.json written")
plt.close("all")


In [ ]:
#@title 📦 EXPORT — results zip + Drive copy + auto-download {display-mode:"form"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

def run_export():
    return colab_kit.export_results_zip_hub(globals())

run_export()

## 🗺️ The full notebook collection

| notebook | what it does |
|---|---|
| [`quickstart_colab.ipynb`](quickstart_colab.ipynb) | 5-minute tour of all 13 disaster domains |
| [`dataset_download_colab.ipynb`](dataset_download_colab.ipynb) | access hub for the 24 registered benchmarks |
| [`checkpoint_zoo_colab.ipynb`](checkpoint_zoo_colab.ipynb) | real weights → real inference (or train tiny in-session) |
| [`demo_colab.ipynb`](demo_colab.ipynb) | single-image demo across tasks |
| [`change_detection_colab.ipynb`](change_detection_colab.ipynb) | binary change detection (pre/post) |
| **`scd_colab.ipynb`** | **semantic change detection — land-cover *from→to* matrices** |
| **`evaluation_metrics_colab.ipynb`** | **mIoU · F1 · confusion matrices for any prediction** |
| [`satellite_finetune_colab.ipynb`](satellite_finetune_colab.ipynb) | finetune on your own labeled pairs |
| [`batch_inference_colab.ipynb`](batch_inference_colab.ipynb) | folder-level batch inference |
| domain notebooks (flood · wildfire · tsunami · hurricane · landslide · volcano) | per-domain guided demos |

Framework: [`SyedRafiAhmedGIS/AnyDisasterMapping`](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping) — paper: *Earth Observation for Disaster Mapping: Benchmarks, Methods, Challenges and Future Perspectives* (SSRN 6725082).
